# Session 3: a turbojet, built and validated
**A. Real-world problem.** A jet engine does not make shaft power. The turbine takes only
enough work to drive the compressor, and the rest of the hot gas expands through a nozzle
and leaves fast: the push on the engine is the thrust. How much thrust does one kilogram
of air per second give, how much fuel does it need, and how do the losses in the compressor,
turbine and nozzle change the answer?

Educational model: air with constant properties, a stationary engine on a test stand
(flight speed 0), fully expanded nozzle, fuel mass ignored. Not a design tool.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
GAMMA, CP = 1.4, 1005.0      # air: ratio of specific heats (no unit), specific heat (J/(kg K))
K = (GAMMA - 1) / GAMMA
LHV = 43e6                   # fuel lower heating value (J/kg), typical for kerosene
ETA_B = 0.98                 # combustor efficiency (no unit)

## Step 1: real compressor and turbine
An isentropic efficiency says how much of the ideal work is really achieved.

    T2 = T1 [1 + (r^K - 1) / eta_c]         compressor needs more work than ideal
    T4 = T3 - (T2 - T1)                     the turbine only drives the compressor
    (p4 / p3)^K = 1 - (T3 - T4) / (eta_t T3)    pressure left after the turbine

Here p3 = r p1 (no pressure loss in the combustor).

In [ ]:
def turbojet(t1, r, t3, eta_c=0.85, eta_t=0.90, eta_n=0.95):
    """Static turbojet. Returns a dictionary in SI units (K, J/kg, m/s, N s/kg).

    Raises ValueError if the turbine cannot leave any pressure for a jet.
    """
    if r < 1:
        raise ValueError("the pressure ratio must be at least 1")
    for e in (eta_c, eta_t, eta_n):
        if not 0 < e <= 1:
            raise ValueError("efficiencies must be above 0 and at most 1")
    t2 = t1 * (1 + (r ** K - 1) / eta_c)
    t4 = t3 - (t2 - t1)
    if t4 <= 0 or t3 <= t2:
        raise ValueError("T3 must be hotter than T2")
    inner = 1 - (t3 - t4) / (eta_t * t3)
    if inner <= 0:
        raise ValueError("the turbine cannot drive the compressor")
    p_ratio_nozzle = r * inner ** (1 / K)      # p4 / p1: the nozzle expands from p4 down to p1
    if p_ratio_nozzle < 1:
        raise ValueError("no pressure is left for a jet: raise T3 or the efficiencies, or lower r")
    t5s = t4 * p_ratio_nozzle ** (-K)
    dt_n = eta_n * (t4 - t5s)
    v_e = (2 * CP * dt_n) ** 0.5
    q_in = CP * (t3 - t2)
    fuel_air = q_in / (LHV * ETA_B)
    return {"t2": t2, "t4": t4, "t5": t4 - dt_n, "p_ratio_nozzle": p_ratio_nozzle, "v_e": v_e,
            "thrust_sp": v_e, "q_in": q_in, "fuel_air": fuel_air, "tsfc": fuel_air / v_e * 1e6 if v_e > 0 else float("inf")}

## Step 2: thrust and fuel use
On a test stand the air enters at rest, so the specific thrust (thrust per kg/s of air) is
just the exit speed. The nozzle turns the temperature drop into speed:

    v_e = sqrt(2 cp eta_n (T4 - T5s))        F / mdot = v_e - 0 (N s/kg)

The fuel needed per kg of air is q_in / (eta_b x LHV). The thrust specific fuel consumption
(TSFC) is the fuel per second per newton of thrust, in grams per kilonewton second.

In [ ]:
def thrust_kn(mdot, **kw):
    """Thrust in kilonewtons for an air flow mdot (kg/s)."""
    return mdot * turbojet(**kw)["thrust_sp"] / 1000


def fuel_flow_kg_s(mdot, **kw):
    return mdot * turbojet(**kw)["fuel_air"]

## Step 3: validation
Two tests that do not need a real engine:

1. With no losses (all efficiencies 1) the kinetic energy of the jet, v_e^2 / 2, must equal the
   net work of the ideal cycle from Session 2: w_net = cp [(T3 - T4) - (T2 - T1)].
2. Energy balance of the whole engine: q_in = cp (T5 - T1) + v_e^2 / 2 (the heat added leaves
   as hot exhaust and as jet kinetic energy).

In [ ]:
def ideal_net_work(t1, r, t3):
    x = r ** K
    return CP * ((t3 - t3 / x) - (t1 * x - t1))

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    kw = dict(t1=288, r=10, t3=1400)
    j = turbojet(**kw)
    print(f"T2 = {j['t2']:.1f} K, T4 = {j['t4']:.1f} K, nozzle pressure ratio = {j['p_ratio_nozzle']:.2f}")
    print(f"exit speed = {j['v_e']:.0f} m/s = specific thrust {j['thrust_sp']:.0f} N s/kg")
    print(f"fuel/air = {j['fuel_air']:.4f}, TSFC = {j['tsfc']:.1f} g/(kN s)")
    print(f"20 kg/s of air -> thrust {thrust_kn(20, **kw):.1f} kN, fuel {fuel_flow_kg_s(20, **kw):.2f} kg/s")
    ideal = turbojet(**kw, eta_c=1, eta_t=1, eta_n=1)
    print(f"no losses: v_e^2/2 = {ideal['v_e'] ** 2 / 2 / 1000:.1f} kJ/kg, Session 2 w_net = {ideal_net_work(288, 10, 1400) / 1000:.1f} kJ/kg")

    r = np.linspace(2, 30, 140)
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    for t3x, col in ((1200, "tab:green"), (1400, "tab:blue"), (1600, "tab:red")):
        ys = []
        for x in r:
            try:
                ys.append(turbojet(288, x, t3x)["thrust_sp"])
            except ValueError:
                ys.append(np.nan)
        ax.plot(r, ys, color=col, label=f"T3 = {t3x} K")
    ax.set_xlabel("pressure ratio r (no unit)")
    ax.set_ylabel("specific thrust (N s per kg of air)")
    ax.set_title("Hotter turbine inlet, more thrust per kg of air")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "thrust_vs_ratio.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6.4, 4.2))
    for e, col in ((1.0, "tab:gray"), (0.9, "tab:blue"), (0.8, "tab:red")):
        ys = []
        for x in r:
            try:
                ys.append(turbojet(288, x, 1400, eta_c=e, eta_t=e, eta_n=min(1.0, e + 0.05))["tsfc"])
            except ValueError:
                ys.append(np.nan)
        ax2.plot(r, ys, color=col, label=f"compressor and turbine efficiency {e}")
    ax2.set_xlabel("pressure ratio r (no unit)")
    ax2.set_ylabel("TSFC (g of fuel per kN per s)")
    ax2.set_ylim(15, 60)
    ax2.set_title("Losses raise the fuel needed per newton")
    ax2.grid(alpha=0.3)
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "tsfc_vs_ratio.png", dpi=120)

## Where this stops
The model has no flight speed or ram compression, no pressure loss in the combustor or duct,
no cooling air for the turbine blades and constant gas properties. The roadmap adds them.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
"""Checkpoint for Session 3. Prints PASS or FAIL."""
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    j = L.turbojet(288, 10, 1400)
    checks.append((f"worked example: T2 = 603.3 K (got {j['t2']:.1f})", abs(j["t2"] - 603.3) < 0.1))
    checks.append((f"worked example: T4 = 1084.7 K (got {j['t4']:.1f})", abs(j["t4"] - 1084.7) < 0.1))
    checks.append((f"worked example: nozzle pressure ratio 3.65 (got {j['p_ratio_nozzle']:.2f})", abs(j["p_ratio_nozzle"] - 3.65) < 0.01))
    checks.append((f"worked example: exit speed 800 m/s (got {j['v_e']:.0f})", abs(j["v_e"] - 800) < 1))
    checks.append((f"worked example: TSFC 23.7 g/(kN s) (got {j['tsfc']:.1f})", abs(j["tsfc"] - 23.7) < 0.1))
    checks.append((f"worked example: 20 kg/s gives 16.0 kN (got {L.thrust_kn(20, t1=288, r=10, t3=1400):.1f})",
                   abs(L.thrust_kn(20, t1=288, r=10, t3=1400) - 16.0) < 0.05))
    # validation 1: no losses -> jet kinetic energy equals the ideal net work of Session 2
    ok1 = True
    for r, t3 in ((5, 1200), (10, 1400), (20, 1600)):
        i = L.turbojet(288, r, t3, eta_c=1, eta_t=1, eta_n=1)
        ok1 &= abs(i["v_e"] ** 2 / 2 - L.ideal_net_work(288, r, t3)) < 1e-6 * L.ideal_net_work(288, r, t3)
    checks.append(("validation: with no losses, v_e^2 / 2 equals the ideal net work of Session 2", ok1))
    # validation 2: energy balance of the whole engine
    ok2 = True
    for r, t3, ec, et, en in ((10, 1400, 0.85, 0.9, 0.95), (16, 1600, 0.8, 0.85, 0.9), (6, 1200, 0.9, 0.9, 1.0)):
        g = L.turbojet(288, r, t3, ec, et, en)
        ok2 &= abs(g["q_in"] - (L.CP * (g["t5"] - 288) + g["v_e"] ** 2 / 2)) < 1e-6 * g["q_in"]
    checks.append(("validation: heat added = enthalpy gain of the exhaust + jet kinetic energy", ok2))
    checks.append(("losses reduce the thrust", L.turbojet(288, 10, 1400, 0.8, 0.85, 0.9)["v_e"] < j["v_e"] < L.turbojet(288, 10, 1400, 1, 1, 1)["v_e"]))
    checks.append(("a hotter turbine inlet gives more specific thrust", L.turbojet(288, 10, 1600)["v_e"] > j["v_e"] > L.turbojet(288, 10, 1200)["v_e"]))
    checks.append(("design: r = 16, T3 = 1600 K reaches 900 N s/kg and 24 g/(kN s)",
                   L.turbojet(288, 16, 1600)["v_e"] >= 900 and L.turbojet(288, 16, 1600)["tsfc"] <= 24))
    checks.append(("edge case: r = 1 gives zero thrust", abs(L.turbojet(288, 1, 1400)["v_e"]) < 1e-9))
    for bad, label in ((lambda: L.turbojet(288, 0.5, 1400), "r below 1"),
                       (lambda: L.turbojet(288, 10, 1400, eta_c=0.0), "compressor efficiency 0"),
                       (lambda: L.turbojet(288, 30, 800), "T3 too low for the compressor"),
                       (lambda: L.turbojet(288, 30, 1000, 0.7, 0.7, 1.0), "no pressure left for a jet")):
        try:
            bad()
            raised = False
        except ValueError:
            raised = True
        checks.append((f"edge case: {label} gives an error", raised))
    return checks
import types
L = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
ok = True
for label, passed in run_checks(L):
    print(('PASS  ' if passed else 'FAIL  ') + label)
    ok &= bool(passed)
print('PASS' if ok else 'FAIL')